# Step 1 — Parsing JSON-like Columns
Mengekstrak nama entitas dari kolom `genres`, `cast`, `crew`, dan `keywords` yang berformat string JSON.

In [1]:
import pandas as pd
import ast
import os

DATA_DIR   = os.path.join('..', 'dataset')
OUTPUT_DIR = os.path.join('..', 'dataset')

## Helper: safe parser

In [2]:
def safe_parse(val):
    """Parse string JSON-like menjadi list of dict."""
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

---
## 1. Parse `genres` dari movies.csv
Ekstrak nama genre saja.

In [3]:
df_movies = pd.read_csv(os.path.join(DATA_DIR, 'movies.csv'))
print(f'Shape awal: {df_movies.shape}')
df_movies.head(2)

Shape awal: (45463, 5)


,id,title,genres,release_date,revenue
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",1995-10-30,373554033.0
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",1995-12-15,262797249.0


In [4]:
# Parse genres → list nama genre
df_movies['genres_parsed'] = df_movies['genres'].apply(
    lambda x: [g['name'] for g in safe_parse(x)]
)

# Hapus kolom genres asli, ganti dengan yang sudah di-parse
df_movies = df_movies.drop(columns=['genres'])
df_movies = df_movies.rename(columns={'genres_parsed': 'genres'})

print(f'Contoh genres row 0: {df_movies["genres"].iloc[0]}')
df_movies.head(3)

Contoh genres row 0: ['Animation', 'Comedy', 'Family']


,id,title,release_date,revenue,genres
0,862,Toy Story,1995-10-30,373554033.0,"[Animation, Comedy, Family]"
1,8844,Jumanji,1995-12-15,262797249.0,"[Adventure, Fantasy, Family]"
2,15602,Grumpier Old Men,1995-12-22,0.0,"[Romance, Comedy]"


In [5]:
df_movies.to_csv(os.path.join(OUTPUT_DIR, 'movies_clean.csv'), index=False)
print('Tersimpan: dataset/movies_clean.csv')

Tersimpan: dataset/movies_clean.csv


---
## 2. Parse `cast` dan `crew` dari credits.csv
- `cast` → ambil nama aktor, maksimal 5 teratas (by order)
- `crew` → filter `job == 'Director'`, ambil nama sutradara

In [6]:
df_credits = pd.read_csv(os.path.join(DATA_DIR, 'credits.csv'))
print(f'Shape awal: {df_credits.shape}')
df_credits.head(2)

Shape awal: (45476, 3)


,id,cast,crew
0,862,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,8844,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."


In [7]:
def extract_actors(val, top_n=5):
    """Ambil nama aktor top N berdasarkan urutan (order)."""
    data = safe_parse(val)
    # Urutkan by 'order' lalu ambil top N
    sorted_cast = sorted(data, key=lambda x: x.get('order', 999))
    return [c['name'] for c in sorted_cast[:top_n]]

def extract_directors(val):
    """Ambil nama sutradara dari crew."""
    data = safe_parse(val)
    return [c['name'] for c in data if c.get('job') == 'Director']

df_credits['actors']    = df_credits['cast'].apply(extract_actors)
df_credits['directors'] = df_credits['crew'].apply(extract_directors)

# Hanya simpan kolom yang diperlukan
df_credits_clean = df_credits[['id', 'actors', 'directors']].copy()

print(f'Shape: {df_credits_clean.shape}')
print(f'Contoh actors row 0   : {df_credits_clean["actors"].iloc[0]}')
print(f'Contoh directors row 0: {df_credits_clean["directors"].iloc[0]}')
df_credits_clean.head(3)

Shape: (45476, 3)
Contoh actors row 0   : ['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim Varney', 'Wallace Shawn']
Contoh directors row 0: ['John Lasseter']


,id,actors,directors
0,862,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",[John Lasseter]
1,8844,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",[Joe Johnston]
2,15602,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",[Howard Deutch]


In [8]:
df_credits_clean.to_csv(os.path.join(OUTPUT_DIR, 'credits_clean.csv'), index=False)
print('Tersimpan: dataset/credits_clean.csv')

Tersimpan: dataset/credits_clean.csv


---
## 3. Parse `keywords` dari keywords.csv
Ekstrak nama keyword saja.

In [9]:
df_keywords = pd.read_csv(os.path.join(DATA_DIR, 'keywords.csv'))
print(f'Shape awal: {df_keywords.shape}')
df_keywords.head(2)

Shape awal: (46419, 2)


,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."


In [10]:
df_keywords['keywords_parsed'] = df_keywords['keywords'].apply(
    lambda x: [k['name'] for k in safe_parse(x)]
)

df_keywords = df_keywords.drop(columns=['keywords'])
df_keywords = df_keywords.rename(columns={'keywords_parsed': 'keywords'})

print(f'Shape: {df_keywords.shape}')
print(f'Contoh keywords row 0: {df_keywords["keywords"].iloc[0]}')
df_keywords.head(3)

Shape: (46419, 2)
Contoh keywords row 0: ['jealousy', 'toy', 'boy', 'friendship', 'friends', 'rivalry', 'boy next door', 'new toy', 'toy comes to life']


,id,keywords
0,862,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,"[board game, disappearance, based on children'..."
2,15602,"[fishing, best friend, duringcreditsstinger, o..."


In [11]:
df_keywords.to_csv(os.path.join(OUTPUT_DIR, 'keywords_clean.csv'), index=False)
print('Tersimpan: dataset/keywords_clean.csv')

Tersimpan: dataset/keywords_clean.csv


---
## Ringkasan Hasil Parsing

In [12]:
print('=== Ringkasan Hasil Parsing ===')
print(f'movies_clean.csv  : {df_movies.shape}  | kolom: {list(df_movies.columns)}')
print(f'credits_clean.csv : {df_credits_clean.shape} | kolom: {list(df_credits_clean.columns)}')
print(f'keywords_clean.csv: {df_keywords.shape} | kolom: {list(df_keywords.columns)}')

print('\n=== Cek Missing Values ===')
for name, df in [('movies_clean', df_movies), ('credits_clean', df_credits_clean), ('keywords_clean', df_keywords)]:
    print(f'{name}: {df.isnull().sum().to_dict()}')

=== Ringkasan Hasil Parsing ===
movies_clean.csv  : (45463, 5)  | kolom: ['id', 'title', 'release_date', 'revenue', 'genres']
credits_clean.csv : (45476, 3) | kolom: ['id', 'actors', 'directors']
keywords_clean.csv: (46419, 2) | kolom: ['id', 'keywords']

=== Cek Missing Values ===
movies_clean: {'id': 0, 'title': 3, 'release_date': 87, 'revenue': 3, 'genres': 0}
credits_clean: {'id': 0, 'actors': 0, 'directors': 0}
keywords_clean: {'id': 0, 'keywords': 0}
